# Customer Retention & RFM Analysis — Cancellation Rate Analysis

RFM measures completed purchase behavior, but ignores cancelled orders entirely, meaning a customer who orders and cancels repeatedly could look identical to a genuinely loyal customer. This notebook calculates a cancellation rate per customer to flag this risk separately from RFM scoring.

## 1. Load Raw Data and Separate Cancelled Orders

Reloading the original raw dataset (not the cleaned version) since cancelled orders were excluded during cleaning, we need them back specifically for this analysis.

In [4]:
import pandas as pd

df = pd.read_excel('/Users/piyushkalra/Downloads/online_retail_II.xlsx', engine='openpyxl')


df_cancelled = df[df['Invoice'].astype(str).str.startswith('C')]
df_cancelled = df_cancelled.dropna(subset=['Customer ID'])

# Excluded 'Manual' stock code rows, same as notebook 1
df_cancelled = df_cancelled[~df_cancelled['StockCode'].astype(str).str.upper().eq('M')]

# Changed the data type of Customer ID to int
df_cancelled['Customer ID'] = df_cancelled['Customer ID'].astype(int)

print("Cancelled order rows with valid Customer ID:", len(df_cancelled))

Cancelled order rows with valid Customer ID: 9615


## 2. Load Genuine Purchases (Cleaned Data)

Using the already-cleaned dataset for genuine completed purchases.

In [2]:
df_final = pd.read_csv('/Users/piyushkalra/Desktop/cleaned_retail_data.csv')
df_final.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,order_value
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085,United Kingdom,83.4
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,81.0
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,81.0
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085,United Kingdom,100.8
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085,United Kingdom,30.0


## 3. Calculate Cancellation Rate per Customer

For each customer, comparing how many orders they cancelled versus how many they completed.

In [9]:
purchase_counts = df_final.groupby('Customer ID')['Invoice'].nunique().rename('purchase_orders')
cancel_counts = df_cancelled.groupby('Customer ID')['Invoice'].nunique().rename('cancelled_orders')

customer_behavior = pd.concat([purchase_counts, cancel_counts], axis=1).fillna(0)
customer_behavior['cancellation_rate'] = customer_behavior['cancelled_orders'] / (customer_behavior['purchase_orders'] + customer_behavior['cancelled_orders'])

customer_behavior.sort_values('cancellation_rate', ascending=False).head(10)

real_customers = customer_behavior[customer_behavior['purchase_orders'] > 0]
real_customers.sort_values('cancellation_rate', ascending=False).head(10)

,purchase_orders,cancelled_orders,cancellation_rate
Customer ID,,,
12770,1.0,5.0,0.833333
16118,1.0,4.0,0.800000
13768,2.0,6.0,0.750000
14893,1.0,3.0,0.750000
17531,2.0,5.0,0.714286
12809,2.0,4.0,0.666667
18187,1.0,2.0,0.666667
12807,1.0,2.0,0.666667
13905,1.0,2.0,0.666667


## 4. Merge Cancellation Rate into RFM Segments

Loading the final segmented RFM table from `03_rfm_segmentation.ipynb` and layering cancellation rate on top as a 4th metric, not folded into RFM itself, since RFM specifically measures completed purchase behavior. This lets us flag customers whose RFM looks strong but who cancel constantly.

In [6]:
rfm_final = pd.read_csv('/Users/piyushkalra/Desktop/rfm_final.csv')

rfm_final['Customer ID'] = rfm_final['Customer ID'].astype(int)
customer_behavior.index = customer_behavior.index.astype(int)

rfm_final = rfm_final.merge(
    customer_behavior[['cancellation_rate']],
    left_on='Customer ID',
    right_index=True,
    how='left'
)

rfm_final['cancellation_rate'] = rfm_final['cancellation_rate'].fillna(0)

rfm_final[['Customer ID', 'segment', 'cancellation_rate']].head(10)

,Customer ID,segment,cancellation_rate
0,12346,At Risk,0.214286
1,12347,Potential,0.000000
2,12348,Lost,0.000000
3,12349,Loyal,0.250000
4,12351,Potential,0.000000
5,12352,Potential,0.000000
6,12353,Potential,0.000000
7,12355,Lost,0.000000
8,12356,Loyal,0.000000
9,12357,Potential,0.000000


## 5. Flag "True Champions" and Check Segment-Level Cancellation Patterns

As a customer scoring as Champions on RFM but cancelling constantly isn't really a champion. Flagging this, then checking whether cancellation behavior differs meaningfully across segments.

In [7]:
rfm_final['true_champion'] = (rfm_final['segment'] == 'Champions') & (rfm_final['cancellation_rate'] < 0.2)

n_champions = (rfm_final['segment'] == 'Champions').sum()
n_true_champions = rfm_final['true_champion'].sum()
print(f"Champions: {n_champions} | True Champions (cancellation rate < 20%): {n_true_champions}")

segment_cancellation = rfm_final.groupby('segment')['cancellation_rate'].mean().sort_values(ascending=False)
segment_cancellation

Champions: 1105 | True Champions (cancellation rate < 20%): 698


segment
Champions    0.148089
At Risk      0.136147
Loyal        0.130011
Lost         0.090117
Potential    0.082569
Name: cancellation_rate, dtype: float64

In [8]:
rfm_final.to_csv('/Users/piyushkalra/Desktop/rfm_final.csv', index=False)
print("rfm_final.csv updated with cancellation_rate and true_champion columns - ready for Power BI")

rfm_final.csv updated with cancellation_rate and true_champion columns - ready for Power BI


## Summary

- Calculated per customer cancellation rate from raw order data, kept separate from RFM math since RFM measures completed purchase value specifically.
- Flagged 'true champion' as Champions-segment customers with cancellation rate under 20%, catching cases where strong RFM scores mask unreliable ordering behavior.
- Compared average cancellation rate across segments to check whether cancellation risk concentrates in specific groups.